# GDELT

## Raw event feed — 15-minute snapshot (GDELT 2.0)

Moved out of `source-exploration.ipynb`. GDELT 2.0's 15-min export has 61 columns and no header row — the file has no way to tell you what's in it, so `V2_COLS` below is the schema, kept by position. `usecols` then only actually loads what we use.

In [2]:
import pandas as pd, requests, io, zipfile
from datetime import datetime, timedelta, timezone

# GDELT 2.0 event export schema, fixed column order (no header row in the file)
V2_COLS = (["GlobalEventID","SQLDATE","MonthYear","Year","FractionDate"]
  + [f"Actor{n}{f}" for n in (1,2) for f in
     ["Code","Name","CountryCode","KnownGroupCode","EthnicCode",
      "Religion1Code","Religion2Code","Type1Code","Type2Code","Type3Code"]]
  + ["IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
     "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone"]
  + [f"{p}Geo_{f}" for p in ("Actor1","Actor2","Action") for f in
     ["Type","FullName","CountryCode","ADM1Code","ADM2Code","Lat","Long","FeatureID"]]
  + ["DATEADDED","SOURCEURL"])

USECOLS = ["GlobalEventID","Actor1Name","Actor2Name","EventRootCode","NumMentions",
           "SOURCEURL","ActionGeo_CountryCode","GoldsteinScale","AvgTone"]

# lastupdate.txt can point at a file that isn't actually up yet — a real gap in
# GDELT's own publishing, not just us being early. Step back 15 min at a time
# until one actually exists, instead of trusting the pointer blindly.
t = datetime.strptime(requests.get("http://data.gdeltproject.org/gdeltv2/lastupdate.txt")
                       .text.split()[2].rsplit("/", 1)[-1][:14], "%Y%m%d%H%M%S")
for attempt in range(5):
    url = t.strftime("http://data.gdeltproject.org/gdeltv2/%Y%m%d%H%M00.export.CSV.zip")
    resp = requests.get(url)
    if resp.status_code == 200:
        break
    t -= timedelta(minutes=15)
else:
    raise RuntimeError("no recent 15-min file found")

z = zipfile.ZipFile(io.BytesIO(resp.content))
df = pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None, names=V2_COLS, usecols=USECOLS, low_memory=False)

# biggest story in this 15-min window, by mention count
df.nlargest(5, "NumMentions")[["Actor1Name","Actor2Name","EventRootCode","NumMentions","SOURCEURL"]]

,Actor1Name,Actor2Name,EventRootCode,NumMentions,SOURCEURL
365,GOVERNMENT,REFORMER,2,48,https://www.southernhighlandnews.com.au/story/...
73,NaN,MEMBER OF PARLIAMENT,3,40,https://www.southernhighlandnews.com.au/story/...
315,PREMIER,NaN,1,40,https://www.southernhighlandnews.com.au/story/...
141,AUSTRALIA,REFORMER,2,32,https://www.southernhighlandnews.com.au/story/...
53,NaN,CHRISTIAN,4,24,https://www.southernhighlandnews.com.au/story/...


### Mood ranking — net cooperation/conflict by country

In [10]:
# roughest/calmest countries in this window, by mean Goldstein score
mood = (df.groupby("ActionGeo_CountryCode")
          .agg(events=("GlobalEventID","size"),
               goldstein=("GoldsteinScale","mean"),
               tone=("AvgTone","mean"))
          .query("events >= 7")
          .sort_values("goldstein"))

print(mood.head(5))   # roughest
print(mood.tail(5))   # calmest

                       events  goldstein      tone
ActionGeo_CountryCode                             
RS                         35  -4.625714 -6.770794
KU                         10  -4.600000 -5.126659
UP                         26  -3.280769 -5.935177
PK                         22  -1.400000 -3.826014
LE                         13  -0.738462 -5.576140
                       events  goldstein      tone
ActionGeo_CountryCode                             
NZ                         27   1.737037  0.528725
KS                          7   2.300000 -0.560083
SF                         39   2.351282 -8.521720
FJ                         11   2.600000  0.257147
CU                          8   3.275000 -7.796102


### Best/worst relationships

6-hour window (24 files) instead of one 15-min snapshot, so country pairs actually have enough events to rank.

In [11]:
from datetime import datetime, timedelta, timezone

PAIR_COLS = ["GlobalEventID","Actor1Name","Actor2Name","Actor1CountryCode",
             "Actor2CountryCode","GoldsteinScale","NumMentions","SOURCEURL"]

t = datetime.now(timezone.utc).replace(second=0, microsecond=0)
t -= timedelta(minutes=t.minute % 15 + 15)

frames = []
for i in range(24):
    url = (t - timedelta(minutes=15*i)).strftime(
        "http://data.gdeltproject.org/gdeltv2/%Y%m%d%H%M00.export.CSV.zip")
    try:
        z = zipfile.ZipFile(io.BytesIO(requests.get(url, timeout=30).content))
        frames.append(pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None,
                                   names=V2_COLS, usecols=PAIR_COLS, low_memory=False))
    except Exception:
        pass
recent = pd.concat(frames)

# only cross-border events, pair sorted so US<->CN and CN<->US collapse into one row
p = recent.dropna(subset=["Actor1CountryCode","Actor2CountryCode"])
p = p[p.Actor1CountryCode != p.Actor2CountryCode].copy()
p["pair"] = [" <-> ".join(sorted(x)) for x in zip(p.Actor1CountryCode, p.Actor2CountryCode)]

board = (p.groupby("pair")
          .agg(events=("GlobalEventID","size"), mood=("GoldsteinScale","mean"))
          .query("events >= 15")
          .sort_values("mood"))

print("WORST RELATIONSHIPS TODAY\n", board.head(10), "\n")
print("BEST RELATIONSHIPS TODAY\n", board.tail(10))

WORST RELATIONSHIPS TODAY
              events      mood
pair                         
RUS <-> UKR     165 -5.261212
PRK <-> RUS      17 -4.123529
IRN <-> ISR      26 -3.453846
SAU <-> YEM      25 -2.516000
CUB <-> USA      27 -1.448148
ISR <-> PSE     126 -1.357143
JOR <-> USA      17 -1.129412
ARG <-> BRA      25 -1.112000
IDN <-> MYS      18 -1.000000
BRA <-> USA      56 -0.930357 

BEST RELATIONSHIPS TODAY
              events      mood
pair                         
IRN <-> IRQ      25  2.020000
IRN <-> QAT      59  2.094915
IRN <-> OMN      85  2.183529
ISR <-> UGA      17  2.341176
GRC <-> USA      16  2.481250
IDN <-> THA      25  2.548000
IRN <-> PAK      37  2.927027
PAK <-> TUR      16  3.000000
GBR <-> USA      33  3.057576
ISR <-> LBN      36  6.038889


In [14]:
# drill into the worst pair — what actually happened
worst = board.index[0]
story = p[p.pair == worst].nlargest(1, "NumMentions").iloc[0]
print(f"\n{worst}: {story.Actor1Name} -> {story.Actor2Name}")
print(story.SOURCEURL)


RUS <-> UKR: UKRAINIAN -> RUSSIA
http://www.strategypage.com/%5Chtmw%5Chtlog%5Carticles%5C2026080453242.aspx


## What is each Canadian city about?

30 days of GDELT 1.0 daily files (one request/day instead of 96) — a first look at whether cities have a distinct topical "fingerprint" relative to the national mix.

In [13]:
from datetime import date, timedelta

# GDELT 1.0 daily schema — no ADM2Code in the geo fields, otherwise same idea as V2_COLS
V1_COLS = (["GlobalEventID","SQLDATE","MonthYear","Year","FractionDate"]
  + [f"Actor{n}{f}" for n in (1,2) for f in
     ["Code","Name","CountryCode","KnownGroupCode","EthnicCode",
      "Religion1Code","Religion2Code","Type1Code","Type2Code","Type3Code"]]
  + ["IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
     "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone"]
  + [f"{p}Geo_{f}" for p in ("Actor1","Actor2","Action") for f in
     ["Type","FullName","CountryCode","ADM1Code","Lat","Long","FeatureID"]]
  + ["DATEADDED","SOURCEURL"])

CITY_COLS = ["ActionGeo_CountryCode","ActionGeo_Type","ActionGeo_FullName","EventRootCode"]

frames = []
for i in range(1, 31):                      # last 30 days
    d = (date.today() - timedelta(days=i)).strftime("%Y%m%d")
    try:
        r = requests.get(f"http://data.gdeltproject.org/events/{d}.export.CSV.zip", timeout=60)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        day = pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None,
                           names=V1_COLS, usecols=CITY_COLS, low_memory=False)
        frames.append(day[(day.ActionGeo_CountryCode == "CA") &
                           (day.ActionGeo_Type == 4)])     # 4 = world city
    except Exception as e:
        print(d, "skipped", e)

In [15]:
ca = pd.concat(frames, ignore_index=True)

ca["city"] = ca.ActionGeo_FullName.str.split(",").str[0].str.strip()
ca["prov"] = ca.ActionGeo_FullName.str.split(",").str[1].str.strip()

CAMEO = {1:"Statement",2:"Appeal",3:"Intent to cooperate",4:"Consult",
         5:"Diplomatic coop",6:"Material coop",7:"Aid",8:"Yield",
         9:"Investigate",10:"Demand",11:"Disapprove",12:"Reject",
         13:"Threaten",14:"Protest",15:"Force posture",16:"Reduce relations",
         17:"Coerce",18:"Assault",19:"Fight",20:"Mass violence"}
ca["action"] = ca.EventRootCode.map(CAMEO)

# fingerprint: for each city, which action types are over-represented vs. the national mix
big = ca.city.value_counts().head(15).index
sub = ca[ca.city.isin(big)]

city_mix = pd.crosstab(sub.city, sub.action, normalize="index")
national = sub.action.value_counts(normalize=True)
lift = (city_mix / national).dropna(axis=1)

for c in big:
    top = lift.loc[c].nlargest(3)
    n = (sub.city == c).sum()
    print(f"{c:<12} n={n:<5} " + " | ".join(f"{k} {v:.1f}x" for k, v in top.items()))

Toronto      n=9179  Mass violence 2.8x | Fight 1.5x | Assault 1.4x
Ottawa       n=7561  Force posture 1.5x | Demand 1.5x | Diplomatic coop 1.2x
Vancouver    n=4108  Protest 1.9x | Coerce 1.6x | Yield 1.4x
Quebec       n=3459  Reduce relations 1.5x | Intent to cooperate 1.3x | Material coop 1.3x
Montreal     n=2677  Assault 1.6x | Investigate 1.6x | Fight 1.4x
Calgary      n=2603  Investigate 1.7x | Protest 1.3x | Intent to cooperate 1.3x
Winnipeg     n=1908  Protest 1.4x | Reduce relations 1.3x | Statement 1.2x
Saskatchewan n=1361  Mass violence 3.8x | Appeal 1.3x | Aid 1.3x
Thunder Bay  n=768   Reduce relations 1.9x | Demand 1.4x | Fight 1.4x
Prince Edward Island n=563   Intent to cooperate 1.7x | Diplomatic coop 1.6x | Consult 1.3x
Sarnia       n=506   Material coop 1.7x | Appeal 1.4x | Statement 1.3x
Brockville   n=471   Protest 2.6x | Aid 1.6x | Diplomatic coop 1.5x
Sudbury      n=362   Force posture 4.1x | Material coop 1.5x | Assault 1.3x
Niagara Falls n=358   Threaten 1.5x | Ai

## Country-day panel — fetch mechanics

Phase 2.2 target: a country-day tone/volume panel for ~5 countries over a few months. Countries: **CA, US, MX** (consistent with Wikipedia — same audience, already-known quirks) plus **UK** (the other "likely audience" country) and **GM/Germany** (the culturally-distinct pick from the Wikipedia country discussion, revisited here since GDELT makes it free — every daily file already contains every country, so widening from 3 to 5 costs zero extra requests, unlike Wikimedia's one-call-per-country).

**Gotcha caught while testing**: GDELT's `CountryCode` fields are **FIPS 10-4**, not ISO 3166-1 — `UK` and `GM`, not `GB`/`DE`. Using the ISO codes silently returned ~0 rows for Germany and near-nothing for "GB" (not a real match) instead of erroring, which would've been an easy miss. CA/US/MX happen to be identical in both standards, so this only bit the 2 new countries.

Before committing to a full backfill: confirm the daily file reliably covers all 5, and get a feel for how much daily volume each one actually has (GDELT's coverage is wire-service driven, so it's not evenly distributed — worth checking before trusting the panel).

In [16]:
COUNTRIES = ["CA", "US", "MX", "UK", "GM"]  # FIPS 10-4, not ISO — see note above

PANEL_COLS = ["ActionGeo_CountryCode", "GoldsteinScale", "AvgTone", "GlobalEventID"]

def fetch_gdelt_day(day: date, usecols: list[str]) -> pd.DataFrame:
    """One GDELT 1.0 daily file -> DataFrame with just the columns we asked for."""
    url = f"http://data.gdeltproject.org/events/{day.strftime('%Y%m%d')}.export.CSV.zip"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    return pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None,
                        names=V1_COLS, usecols=usecols, low_memory=False)


# last 5 days, all 5 countries at once — same file has everyone in it
for i in range(1, 6):
    d = date.today() - timedelta(days=i)
    day_df = fetch_gdelt_day(d, PANEL_COLS)
    counts = day_df[day_df.ActionGeo_CountryCode.isin(COUNTRIES)].groupby("ActionGeo_CountryCode").size()
    print(d, dict(counts))

2026-08-03 {'CA': np.int64(1355), 'GM': np.int64(497), 'MX': np.int64(491), 'UK': np.int64(5204), 'US': np.int64(27384)}
2026-08-02 {'CA': np.int64(1174), 'GM': np.int64(406), 'MX': np.int64(213), 'UK': np.int64(3213), 'US': np.int64(15316)}
2026-08-01 {'CA': np.int64(1537), 'GM': np.int64(508), 'MX': np.int64(360), 'UK': np.int64(3192), 'US': np.int64(21248)}
2026-07-31 {'CA': np.int64(2618), 'GM': np.int64(695), 'MX': np.int64(622), 'UK': np.int64(5374), 'US': np.int64(34269)}
2026-07-30 {'CA': np.int64(2893), 'GM': np.int64(659), 'MX': np.int64(460), 'UK': np.int64(5283), 'US': np.int64(37173)}


## Country-day tone/volume panel

The actual deliverable for this checklist item: one row per country per day — event count, mean Goldstein, mean tone — over the last ~3 months. One request per day (not per country), same as the mechanics check above.

In [ ]:
PANEL_DAYS = 90  # ~3 months, "a few months" per the plan

rows = []
for i in range(1, PANEL_DAYS + 1):
    d = date.today() - timedelta(days=i)
    try:
        day_df = fetch_gdelt_day(d, PANEL_COLS)
    except Exception as e:
        print(d, "skipped", e)
        continue
    sub = day_df[day_df.ActionGeo_CountryCode.isin(COUNTRIES)]
    daily = (sub.groupby("ActionGeo_CountryCode")
                .agg(events=("GlobalEventID", "size"),
                     goldstein=("GoldsteinScale", "mean"),
                     tone=("AvgTone", "mean")))
    daily["date"] = d
    rows.append(daily.reset_index())

panel = pd.concat(rows, ignore_index=True).rename(columns={"ActionGeo_CountryCode": "country"})
panel = panel.sort_values(["country", "date"]).reset_index(drop=True)

print(f"{len(panel)} country-days across {panel.country.nunique()} countries, {panel.date.min()} to {panel.date.max()}")
panel.groupby("country")["events"].describe()[["mean", "min", "max"]]

## First derived observation — rolling baseline + deviation flags

The actual point of the panel: is a country's mood on a given day unusual *for that country*, not just in absolute terms — the "relative over raw" idea from the observation-spine model, made real.

Baseline for day D uses only the 14 days strictly before D (`shift(1)`) — otherwise today's own value would leak into its own baseline and dampen every deviation.

In [7]:
WINDOW = 14

by_country = panel.groupby("country")["tone"]
panel["tone_baseline"] = by_country.transform(lambda s: s.rolling(WINDOW, min_periods=WINDOW).mean().shift(1))
panel["tone_std"] = by_country.transform(lambda s: s.rolling(WINDOW, min_periods=WINDOW).std().shift(1))
panel["tone_z"] = (panel["tone"] - panel["tone_baseline"]) / panel["tone_std"]

# |z| > 1.5 -> today's mood is a real outlier relative to this country's own recent baseline
flagged = panel[panel.tone_z.abs() > 1.5].sort_values("tone_z")
flagged[flagged['country'] == "CA"][["country", "date", "tone", "tone_baseline", "tone_z"]]

NameError: name 'panel' is not defined